# 03 — Model Evaluation & Comparison

Use the `AnomalyShield` orchestrator to run all detectors, compare metrics,
build ensemble predictions, and generate a final report.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import TimeSeriesLoader
from src.data.preprocessor import Preprocessor
from src.detector import AnomalyShield
from src.models.isolation_forest import IsolationForestDetector
from src.models.lof import LOFDetector
from src.models.elliptic_envelope import EllipticEnvelopeDetector
from src.utils import set_random_seeds, evaluate_detector, generate_report

set_random_seeds(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load and preprocess data
df_raw = pd.read_csv("../assets/sample_data.csv", parse_dates=["date"])
df_raw = df_raw.set_index("date").sort_index()

df = TimeSeriesLoader.from_csv("../assets/sample_data.csv")
df_scaled, scaler = Preprocessor.normalize(df, method="standard")
X = df_scaled[["value"]].values

# Ground truth: is_anomaly 1 -> -1 (anomaly), 0 -> 1 (normal)
y_true = np.where(df_raw["is_anomaly"] == 1, -1, 1)

print(f"Data points: {len(X)}")
print(f"Ground-truth anomalies: {(y_true == -1).sum()}")

In [ ]:
# Build the AnomalyShield orchestrator with all sklearn-compatible detectors.
# Note: The Autoencoder and Prophet have different APIs (windowed output,
# DataFrame input) so they are not directly compatible with the orchestrator's
# numpy-based run_all(). We evaluate them separately below.

shield = AnomalyShield()

shield.add_detector(IsolationForestDetector(
    name="IsolationForest",
    n_estimators=100,
    contamination=0.05,
    random_state=42,
))

shield.add_detector(LOFDetector(
    name="LOF",
    n_neighbors=20,
    contamination=0.05,
    novelty=True,
))

shield.add_detector(EllipticEnvelopeDetector(
    name="EllipticEnvelope",
    contamination=0.05,
    random_state=42,
))

# Run all detectors with ground truth for automatic metric computation
results = shield.run_all(X, y_true=y_true)

print("Detectors run:", list(results.keys()))

In [ ]:
# Display the comparison table of metrics
comparison_df = shield.compare()
print("Metrics Comparison Table:")
display(comparison_df.round(4))

# Visualize metrics as a grouped bar chart
comparison_df.plot(kind="bar", figsize=(10, 5), edgecolor="white")
plt.title("Detector Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Ensemble predictions using the orchestrator

# Majority vote: anomaly if more than half the detectors flag it
ens_majority = shield.get_ensemble_predictions(strategy="majority")
metrics_majority = evaluate_detector(y_true, ens_majority)

# Unanimous vote: anomaly only if ALL detectors flag it
ens_unanimous = shield.get_ensemble_predictions(strategy="unanimous")
metrics_unanimous = evaluate_detector(y_true, ens_unanimous)

# Any vote: anomaly if ANY detector flags it
ens_any = shield.get_ensemble_predictions(strategy="any")
metrics_any = evaluate_detector(y_true, ens_any)

ensemble_df = pd.DataFrame(
    {
        "Majority": metrics_majority,
        "Unanimous": metrics_unanimous,
        "Any": metrics_any,
    }
).T
ensemble_df.index.name = "strategy"

print("Ensemble Strategy Comparison:")
display(ensemble_df.round(4))

# Anomaly counts per strategy
print(f"\nMajority vote anomalies: {(ens_majority == -1).sum()}")
print(f"Unanimous vote anomalies: {(ens_unanimous == -1).sum()}")
print(f"Any vote anomalies: {(ens_any == -1).sum()}")
print(f"Ground truth anomalies: {(y_true == -1).sum()}")

In [ ]:
# Generate a Markdown report
report = generate_report(results, output_path="../reports/detection_report.md")
print(report)

## Summary and Conclusions

### Individual Detectors

- **Isolation Forest**: Good general-purpose detector. Works well on global outliers with no distributional assumptions.
- **LOF (Local Outlier Factor)**: Density-based approach that excels at detecting local anomalies in varying-density regions.
- **Elliptic Envelope**: Assumes Gaussian distribution. Effective when inlier data is approximately normally distributed.

### Ensemble Strategies

- **Majority vote**: Balanced trade-off between precision and recall. Flags a point only when most detectors agree.
- **Unanimous vote**: High precision, lower recall. Use when false positives are costly.
- **Any vote**: High recall, lower precision. Use when missing anomalies is costly.

### Recommendations

1. Start with the **majority vote ensemble** as a robust baseline.
2. Tune the `contamination` parameter per detector based on expected anomaly rate.
3. For production, add the **LSTM Autoencoder** (from notebook 02) for temporal pattern detection and **Prophet** for seasonality-aware forecasting.
4. Monitor model drift: retrain periodically as the data distribution evolves.
5. Evaluate on held-out test data (time-based split) before deploying to production.